In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = './dbt_demo/service-account.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/03/09 15:01:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
import glob
import os

local_dirs = sorted(glob.glob('data/pq/yellow_tripdata_*'))

for d in local_dirs:
    month_name = os.path.basename(d)
    print(f"Uploading {month_name}...")
    df = spark.read.parquet(f"{d}/{month_name}")
    df.write.parquet(f"gs://gcs_spark_week6/pq/{month_name}/", mode='overwrite')
    print(f"  Done: {month_name}")

print("All yellow data uploaded!")

Uploading yellow_tripdata_2020-01...


  Done: yellow_tripdata_2020-01
Uploading yellow_tripdata_2020-02...


  Done: yellow_tripdata_2020-02
Uploading yellow_tripdata_2020-03...


  Done: yellow_tripdata_2020-03
Uploading yellow_tripdata_2020-04...


  Done: yellow_tripdata_2020-04
Uploading yellow_tripdata_2020-05...


...

  Done: yellow_tripdata_2021-04
Uploading yellow_tripdata_2021-05...


  Done: yellow_tripdata_2021-05
Uploading yellow_tripdata_2021-06...


  Done: yellow_tripdata_2021-06
Uploading yellow_tripdata_2021-07...


  Done: yellow_tripdata_2021-07
Uploading yellow_tripdata_2021-08...


  Done: yellow_tripdata_2021-08
All yellow data uploaded!


In [ ]:
import subprocess

result = subprocess.run(
    ['gsutil', 'ls', '-r', 'gs://gcs_spark_week6/pq/'],
    capture_output=True, text=True
)

yellow_paths = list(set(
    p.strip().rsplit('/', 1)[0] + '/'
    for p in result.stdout.splitlines()
    if 'yellow' in p and p.endswith('.parquet')
))

df_yellow = spark.read.parquet(*yellow_paths)


In [6]:
df_yellow.count()

2304517

In [ ]:
#All these steps are how to connect your local spark cluster to your google storage